In [1]:
suppressPackageStartupMessages({
    library(ArchR) 
    library(data.table)
    library(purrr)
    library(parallel)
    library(dplyr)
    library(Matrix)
    library(gridExtra)
})
options(repr.plot.width=15, repr.plot.height=8)

In [2]:
# I/O
io = list()
io$basedir='/rds/project/rds-SDzz0CATGms/users/bt392/04_Rabbit_ATAC'
io$output.directory <- file.path(io$basedir,"ArchR")
io$plotdir = file.path(io$basedir,'celltype_score')
setwd(io$output.directory)

In [3]:
io$archR.directory = file.path(io$output.directory, 'Project/')

ArchRProject.filt = loadArchRProject(io$archR.directory)

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .____

In [4]:
# Calculate impute weights
ArchRProject.filt = addImputeWeights(
  ArchRProj = ArchRProject.filt,
  reducedDims = "IterativeLSI_Harmony")

ArchR logging to : ArchRLogs/ArchR-addImputeWeights-2a4f0ca1c05c-Date-2021-12-01_Time-18-24-35.log
If there is an issue, please report to github with logFile!

2021-12-01 18:24:36 : Computing Impute Weights Using Magic (Cell 2018), 0 mins elapsed.

Warning message in sprintf("Completed Getting Magic Weights!", round(object.size(weightList)/10^9, :
“one argument not used by format 'Completed Getting Magic Weights!'”


In [4]:
# Get gene score matrix
gene_matrix = getMatrixFromProject(
  ArchRProj = ArchRProject.filt,
  useMatrix = "GeneScoreMatrix")

# Rename columns & rows
gene_names = gene_matrix@elementMetadata$name
gene_matrix = gene_matrix@assays@data$GeneScoreMatrix
rownames(gene_matrix) = gene_names

ArchR logging to : ArchRLogs/ArchR-getMatrixFromProject-1ea885dd2c384-Date-2021-12-05_Time-16-51-26.log
If there is an issue, please report to github with logFile!

2021-12-05 16:54:11 : Organizing colData, 2.746 mins elapsed.

2021-12-05 16:54:11 : Organizing rowData, 2.748 mins elapsed.

2021-12-05 16:54:11 : Organizing rowRanges, 2.748 mins elapsed.

2021-12-05 16:54:11 : Organizing Assays (1 of 1), 2.749 mins elapsed.

2021-12-05 16:54:31 : Constructing SummarizedExperiment, 3.08 mins elapsed.

2021-12-05 16:54:31 : Finished Matrix Creation, 3.087 mins elapsed.



In [6]:
plot_celltype_score = function(markernr){
    # Load markers
    markers = fread("/rds/project/rds-SDzz0CATGms/users/bt392/04_Rabbit_ATAC/RNA/markers.csv", header=TRUE)[,-1]
    markers = head(markers, markernr)

    # get clusters
    meta = ArchRProject.filt@cellColData['Clusters']

    # Calculate celltype scores
    celltype_scores = lapply(colnames(markers), function(x){
        genes = markers[[x]]
        if(length(rownames(gene_matrix)[rownames(gene_matrix) %in% genes]) > 1){
            imputed_matrix = suppressMessages(imputeMatrix(mat = gene_matrix[rownames(gene_matrix) %in% genes,], 
                                     imputeWeights = getImputeWeights(ArchRProject.filt)))

            v1 = scale(t(imputed_matrix))
            score = Matrix::rowSums(v1)
            return(score)
            }
        else{
            score = rep(0, dim(gene_matrix)[2])
            return(score)
        }
    })

    # rename columns
    celltype_scores = t(do.call(rbind.data.frame, celltype_scores))
    colnames(celltype_scores) = gsub(' ', '_', colnames(markers))
    colnames(celltype_scores) = gsub('/', '_', colnames(celltype_scores))
    colnames(celltype_scores) = gsub('-', '_', colnames(celltype_scores))
    rownames(celltype_scores) = colnames(gene_matrix)

    # plot
    options(repr.plot.width=15, repr.plot.height=4)

    per_cluster = list()
    per_celltype = list()

    for(i in 1:length(colnames(celltype_scores))){
        plot = as.data.frame(celltype_scores[,i])
        colnames(plot) = 'celltype'
        plot = merge(plot, meta, by=0)

    per_celltype[[i]] = ggplot(as.data.frame(plot), aes(Clusters, celltype, fill = Clusters)) + 
                    scale_x_discrete(labels = paste0('C', 1:length(unique(plot$Clusters)))) +
                    geom_violin() +
                    geom_boxplot(fill='white', width=0.2) + 
                    ylab('Celltype Score') + 
                    theme_bw() + theme(legend.position='none') +
                    ggtitle(colnames(celltype_scores)[i]) 
    }


    meta = as.data.frame(meta)
    meta$cell = rownames(meta)
    
    for(i in 1:length(unique(meta$Clusters))){
        cells = meta[meta$Clusters==unique(meta$Clusters)[i], ]$cell
        scores_percelltype = celltype_scores[cells, ]
        scores_percelltype = reshape2::melt(scores_percelltype)


        per_cluster[[i]] = ggplot(scores_percelltype, aes(Var2, value, fill=Var2)) + 
            geom_violin(width=1.5) + 
            geom_boxplot(fill='white', width=0.15) + 
            xlab('') + ylab('Celltype Score') + 
            ggtitle(unique(meta$Clusters)[i]) + 
            theme_bw() + theme(legend.position='none', axis.text.x=element_text(angle=-90, hjust=0))
    }

    # Save
    outfile <- sprintf("%s/per_cluster_%s_markers.pdf",io$plotdir, markernr)
    pdf(outfile, width=15, height=5)
        print(per_cluster)
    dev.off()

    outfile <- sprintf("%s/per_celltype_%s_markers.pdf",io$plotdir, markernr)
    pdf(outfile, width=7, height=5)
        print(per_celltype)
    dev.off()
    
    write.csv(celltype_scores, file.path(io$plotdir,'celltype_score.csv'))
}

In [7]:
sapply(c(2,5,10,20,50,100), plot_celltype_score)